In [ ]:
import polars as pl
import numpy as np
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
from datetime import timedelta, datetime
import warnings
import yfinance as yf
import gc

# Suppress warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. CONFIGURATION
# ==========================================
class Config:
    # Paths
    DATA_PATH = "D:\\MASTER THESIS DATA\\bitmex\\processed_5m_new\\"
    FUNDING_PATH = "D:\\MASTER THESIS DATA\\bitmex\\funding_rates-2025-11-01_2023-01-01\\"
    
    START_DATE = "2023-01-01"
    END_DATE = "2025-11-01"
    
    MAX_MISSING_PCT = 0.05
    CORR_THRESHOLD = 0.90
    MAX_PAIRS = 10
    TC_BPS = 10
    
    # Circuit Breaker: Freezes Execution, BUT we still take PnL hits.
    USE_OUTLIER_FILTER = True 

# --- PARAMETER SWEEP SETTINGS ---
TEST_CASES = [
    {"name": "Fast (4h Session)",  "l_freq": "4h",  "window_l": 14, "window_y": 3},
    {"name": "Medium (8h Session)", "l_freq": "8h",  "window_l": 30, "window_y": 7},
    {"name": "Slow (12h Session)", "l_freq": "12h", "window_l": 60, "window_y": 14},
    {"name": "Daily (24h Session)", "l_freq": "24h", "window_l": 100, "window_y": 30},
]

# ==========================================
# 2. DATA HANDLERS
# ==========================================
class FastDataHandler:
    def __init__(self, folder_path):
        self.folder_path = folder_path
        self.df_wide = None

    def load_data(self):
        print(f"Loading 5m bars from {self.folder_path}...")
        all_files = glob.glob(os.path.join(self.folder_path, "**", "*.parquet"), recursive=True)
        if not all_files: raise FileNotFoundError("No parquet files found.")
        
        q = pl.scan_parquet(all_files).with_columns(pl.col("timestamp").cast(pl.Datetime))
        
        self.df_wide = (
            q.collect()
            .pivot(index="timestamp", on="symbol", values="mid_price", aggregate_function="last")
            .sort("timestamp")
        )
        
        self.df_wide = self.df_wide.filter(
            (pl.col("timestamp") >= datetime.strptime(Config.START_DATE, "%Y-%m-%d")) &
            (pl.col("timestamp") <= datetime.strptime(Config.END_DATE, "%Y-%m-%d"))
        )

        start = self.df_wide["timestamp"].min(); end = self.df_wide["timestamp"].max()
        print(f"Data Loaded: {start} to {end}")
        
        grid = pl.datetime_range(start, end, interval="5m", eager=True).alias("timestamp").to_frame()
        target_dtype = self.df_wide["timestamp"].dtype
        grid = grid.with_columns(pl.col("timestamp").cast(target_dtype))
        
        self.df_wide = grid.join(self.df_wide, on="timestamp", how="left")
        self.df_wide = self.df_wide.with_columns(pl.all().forward_fill(limit=3))
        print(f"Matrix Shape: {self.df_wide.shape}")

    def get_slice(self, start_time, end_time):
        return self.df_wide.filter((pl.col("timestamp") >= start_time) & (pl.col("timestamp") <= end_time))

class FundingHandler:
    def __init__(self, folder_path):
        self.folder_path = folder_path
        self.df_funding = None

    def load_rates(self):
        print(f"Loading Funding Rates...")
        try:
            q = pl.scan_csv(os.path.join(self.folder_path, "*.csv"))
            self.df_funding = (
                q.with_columns([
                    pl.col("timestamp")
                      .str.replace("T", " ")
                      .str.replace("Z", "")
                      .str.to_datetime(strict=False)
                ])
                .drop_nulls(subset=["timestamp"])
                .select(["timestamp", "symbol", "fundingRate"])
                .collect()
                .pivot(index="timestamp", on="symbol", values="fundingRate", aggregate_function="last")
                .sort("timestamp")
            )
            print(f"Funding Loaded. Shape: {self.df_funding.shape}")
        except Exception as e:
            print(f"Error loading funding: {e}")

    def get_rates_slice(self, start_time, end_time):
        if self.df_funding is None: return None
        return self.df_funding.filter((pl.col("timestamp") >= start_time) & (pl.col("timestamp") <= end_time))

# ==========================================
# 3. MATH ENGINE
# ==========================================
class OUCalibrator:
    @staticmethod
    def fit(spread, dt=1.0, strict=True):
        n = len(spread)
        if n < 10: return {'success': False}
        x = spread[:-1]; y = spread[1:]
        sum_x = np.sum(x); sum_y = np.sum(y)
        sum_xx = np.sum(x*x); sum_xy = np.sum(x*y)
        count = n - 1
        denom = (count * sum_xx - sum_x * sum_x)
        if denom == 0: return {'success': False}
        
        b = (count * sum_xy - sum_x * sum_y) / denom
        a = (sum_y - b * sum_x) / count
        
        if strict:
            if b >= 1.0 or b <= 0.0: return {'success': False}
        else:
            if b >= 1.0: b = 0.999 
            if b <= 0.0: return {'success': False}
        
        residuals = y - (a + b * x)
        std_resid = np.std(residuals)
        theta = -np.log(b) / dt
        mu = a / (1 - b)
        term = 1 - b**2
        sigma = std_resid if term <= 1e-8 else std_resid * np.sqrt((2 * theta) / term)
        
        return {'theta': theta, 'mu': mu, 'sigma': sigma, 'success': True}

# ==========================================
# 4. STRATEGY ENGINE
# ==========================================
class StrategyEngine:
    def __init__(self, data_handler, funding_handler, params_name, l_freq, window_l, window_y):
        self.dh = data_handler
        self.fh = funding_handler
        self.name = params_name
        self.l_freq = l_freq
        self.window_l = window_l
        self.window_y = window_y
        self.equity_curve = [] 

    def calculate_ou_variance(self, theta, sigma, dt):
        if theta < 1e-5: return float('inf')
        return (sigma**2 / (2*theta)) * (1 - np.exp(-2 * theta * dt))

    def get_adjustment_factor(self, symbol, log_prices_array, current_idx, position_arr):
        if "USDT" not in symbol: return 1.0 
        current_pos = position_arr[current_idx]
        entry_idx = current_idx
        for i in range(current_idx - 1, -1, -1):
            if position_arr[i] != current_pos:
                entry_idx = i + 1
                break
        
        log_ent = log_prices_array[entry_idx]
        log_cur = log_prices_array[current_idx]
        
        if log_ent < -10: return 0.0
            
        ratio = np.exp(log_cur - log_ent)
        if ratio > 5.0: ratio = 5.0 
        return ratio

    def run(self):
        print(f"\n[RUNNING] {self.name} | Freq: {self.l_freq} | WinL: {self.window_l}d")
        
        print("  > Pre-calculating Trend Bars...", end=" ", flush=True)
        try:
            df_L_full = (
                self.dh.df_wide.lazy()
                .group_by_dynamic("timestamp", every=self.l_freq)
                .agg(pl.all().last())
                .collect()
            )
            print("Done.")
        except Exception as e:
            print(f"Error: {e}")
            return pd.DataFrame()

        trend_dates = df_L_full["timestamp"].to_list()
        
        start_idx = 0
        min_hist = timedelta(days=self.window_l)
        for i, d in enumerate(trend_dates):
            if d - trend_dates[0] > min_hist:
                start_idx = i
                break
        
        try: hour_val = int(self.l_freq.replace("h", "")); dt_L_val = hour_val / 24.0
        except: dt_L_val = 1/3
            
        cumulative_pnl = 0.0
        total_capital_units = Config.MAX_PAIRS * 2
        all_cols = self.dh.df_wide.columns
        total_sessions = len(trend_dates) - 1 - start_idx
        
        for i in range(start_idx, len(trend_dates)-1):
            curr_time = trend_dates[i]
            next_time = trend_dates[i+1]
            
            # Bankruptcy Check
            if (cumulative_pnl / total_capital_units) <= -0.95:
                remaining = len(trend_dates) - 1 - i
                for _ in range(remaining): self.equity_curve.append({'time': trend_dates[i+_], 'pnl': cumulative_pnl})
                break

            step = i - start_idx + 1
            if step % 50 == 0 or step == total_sessions:
                print(f"  Processing {step}/{total_sessions} | PnL: {cumulative_pnl:.2f}", end='\r')

            l_start = curr_time - timedelta(days=self.window_l)
            df_L_slice = df_L_full.filter((pl.col("timestamp") >= l_start) & (pl.col("timestamp") <= curr_time))
            y_start = curr_time - timedelta(days=self.window_y)
            df_Y = self.dh.get_slice(y_start, curr_time)
            df_test = self.dh.get_slice(curr_time, next_time) 
            
            df_fund = None
            if self.fh.df_funding is not None: df_fund = self.fh.get_rates_slice(curr_time, next_time)

            if df_L_slice.height < 5: continue
            
            null_counts = df_L_slice.null_count()
            rows = df_L_slice.height
            valid_syms = [c for c in all_cols if c!="timestamp" and (null_counts[c][0]/rows < Config.MAX_MISSING_PCT)]
            
            if df_test.height > 0:
                nulls_test = df_test.null_count()
                valid_syms = [c for c in valid_syms if nulls_test[c][0] == 0]

            if len(valid_syms) < 2 or df_test.height == 0: continue

            try:
                df_corr = df_Y.select(valid_syms).fill_null(strategy="forward").drop_nulls()
                if df_corr.height < 50: continue
                corr_vals = df_corr.corr().to_numpy()
                syms = df_corr.columns
                candidates = []
                for r in range(len(syms)):
                    for c in range(r+1, len(syms)):
                        if corr_vals[r, c] > Config.CORR_THRESHOLD: candidates.append((syms[r], syms[c]))
                
                ranked_metrics = []

                for sym_a, sym_b in candidates[:50]:
                    s_a = df_L_slice[sym_a].to_numpy()
                    s_b = df_L_slice[sym_b].to_numpy()
                    mask = ~np.isnan(s_a) & ~np.isnan(s_b)
                    if np.sum(mask) < 15: continue
                    L_spread = np.log(s_a[mask]) - np.log(s_b[mask])
                    
                    params_L = OUCalibrator.fit(L_spread, dt=dt_L_val, strict=False)
                    if not params_L['success']: continue
                    
                    y_a = df_corr[sym_a].to_numpy()
                    y_b = df_corr[sym_b].to_numpy()
                    params_Y = OUCalibrator.fit(np.log(y_a) - np.log(y_b), dt=1/288, strict=True)
                    if not params_Y['success']: continue
                    
                    var_L = self.calculate_ou_variance(params_L['theta'], params_L['sigma'], dt=dt_L_val)
                    var_Y = self.calculate_ou_variance(params_Y['theta'], params_Y['sigma'], dt=1/288)
                    ranked_metrics.append({'pair': (sym_a, sym_b), 'var_L': var_L, 'var_Y': var_Y, 'spread_L': L_spread})

                if not ranked_metrics: continue

                met = pd.DataFrame(ranked_metrics)
                met['score'] = met['var_L'].rank(ascending=True) + met['var_Y'].rank(ascending=False)
                top_pairs = met.sort_values('score').head(Config.MAX_PAIRS)

                session_pnl = 0
                df_t_raw = df_test.select(["timestamp"] + valid_syms).fill_null(strategy="forward")
                timestamps_safe = df_t_raw["timestamp"].cast(pl.Int64).to_numpy()
                target_dtype = df_t_raw["timestamp"].dtype

                for _, row in top_pairs.iterrows():
                    sym_a, sym_b = row['pair']
                    L_hist = row['spread_L']
                    
                    epsilon = np.percentile(np.abs(np.diff(L_hist)), 98)
                    
                    dynamic_limit = 5.0 * epsilon
                    HARD_CAP = 0.05 
                    outlier_threshold = min(dynamic_limit, HARD_CAP)
                    if outlier_threshold < 0.005: outlier_threshold = 0.005

                    raw_a = df_t_raw[sym_a].to_numpy()
                    raw_b = df_t_raw[sym_b].to_numpy()
                    
                    def prepare_death(arr):
                        filled = pd.Series(arr).ffill().to_numpy()
                        if len(filled) > 0 and np.isnan(filled[-1]):
                            valid = np.where(~np.isnan(filled))[0]
                            if len(valid) > 0: filled[valid[-1]+1:] = 1e-5
                            else: filled[:] = 1e-5
                        return np.nan_to_num(filled, nan=1e-5)

                    clean_a = prepare_death(raw_a)
                    clean_b = prepare_death(raw_b)
                    
                    t_a = np.log(np.maximum(clean_a, 1e-8))
                    t_b = np.log(np.maximum(clean_b, 1e-8))
                    Y_spread = t_a - t_b
                    
                    target = L_hist[-1]
                    upper, lower = target + epsilon, target - epsilon
                    pos = np.zeros(len(Y_spread))
                    curr = 0
                    
                    for t in range(len(Y_spread)):
                        v = Y_spread[t]
                        
                        # EXECUTION GATING
                        # If raw data is missing, we CANNOT execute. Hold.
                        is_missing = np.isnan(raw_a[t]) or np.isnan(raw_b[t])
                        if is_missing:
                            pos[t] = curr; continue

                        # CIRCUIT BREAKER
                        # If volatility is high, we CANNOT execute. Hold.
                        if Config.USE_OUTLIER_FILTER and t > 0:
                            if abs(v - Y_spread[t-1]) > outlier_threshold:
                                pos[t] = curr; continue 

                        # ZOMBIE GUARD
                        if curr == 0:
                            if clean_a[t] < 1e-4 or clean_b[t] < 1e-4:
                                pos[t] = 0; continue

                        if curr == 0:
                            if v > upper: curr = -1
                            elif v < lower: curr = 1
                        elif curr == -1:
                            if v < target: curr = 0
                        elif curr == 1:
                            if v > target: curr = 0
                        pos[t] = curr
                    
                    # --- PNL CALCULATION ---
                    d_spread = np.diff(Y_spread, prepend=Y_spread[0])
                    d_spread = np.nan_to_num(d_spread)
                
                    
                    pos_active = np.roll(pos, 1); pos_active[0] = 0
                    
                    # Raw PnL
                    raw_pnl = pos_active * d_spread
                    # Liquidation Limits
                    raw_pnl = np.maximum(raw_pnl, -1.0)
                    mask_short_win = (pos_active < 0) & (raw_pnl > 1.0)
                    raw_pnl[mask_short_win] = 1.0 
                    
                    gross = np.sum(raw_pnl)
                    
                    if pos_active[-1] != 0:
                        trades = np.sum(np.abs(np.diff(np.append(pos_active, 0), prepend=0)))
                    else:
                        trades = np.sum(np.abs(np.diff(pos_active, prepend=0)))
                    cost = trades * (Config.TC_BPS/10000)
                    
                    f_pnl = 0.0
                    if df_fund is not None and df_fund.height > 0:
                        for f_row in df_fund.iter_rows(named=True):
                            try:
                                f_ts = f_row['timestamp']
                                f_ts_int = pl.select(pl.lit(f_ts).cast(target_dtype).cast(pl.Int64)).item()
                                idx = np.searchsorted(timestamps_safe, f_ts_int)
                                if idx < len(timestamps_safe) and timestamps_safe[idx] == f_ts_int:
                                    p = pos_active[idx]
                                    if p != 0 and clean_a[idx] > 1e-4 and clean_b[idx] > 1e-4:
                                        ra = f_row.get(sym_a, 0.0) or 0.0
                                        rb = f_row.get(sym_b, 0.0) or 0.0
                                        adj_a = self.get_adjustment_factor(sym_a, t_a, idx, pos_active)
                                        adj_b = self.get_adjustment_factor(sym_b, t_b, idx, pos_active)
                                        adj_a = min(adj_a, 5.0)
                                        adj_b = min(adj_b, 5.0)
                                        f_pnl += p * ((rb * adj_b) - (ra * adj_a))
                            except: pass
                    
                    session_pnl += (gross - cost + f_pnl)

                cumulative_pnl += session_pnl
                self.equity_curve.append({'time': curr_time, 'pnl': cumulative_pnl})

            except Exception as e:
                pass
        
        print(f"\n[DONE] Strategy {self.name} finished.")
        return pd.DataFrame(self.equity_curve)

# ==========================================
# 5. METRICS & PLOTTING
# ==========================================
def calculate_metrics(df_res, capital):
    df = df_res.clone() if isinstance(df_res, pl.DataFrame) else df_res.copy()
    df['roi'] = (df['pnl'] / capital) * 100
    df['date'] = df['time'].dt.date
    daily_equity = df.groupby('date')['roi'].last()
    daily_rets = daily_equity.diff().fillna(0)
    
    mean_ret = daily_rets.mean()
    std_ret = daily_rets.std()
    sharpe = (mean_ret / std_ret) * np.sqrt(365) if std_ret != 0 else 0
    
    neg_rets = daily_rets[daily_rets < 0]
    downside_std = neg_rets.std()
    sortino = (mean_ret / downside_std) * np.sqrt(365) if downside_std != 0 else 0
    
    wealth_index = 1 + (df['roi'] / 100)
    running_max = wealth_index.cummax()
    drawdown = (wealth_index - running_max) / running_max
    max_dd = drawdown.min() * 100

    return {"Total Return": df['roi'].iloc[-1], "Sharpe": sharpe, "Sortino": sortino, "MaxDD": max_dd}

if __name__ == "__main__":
    if not os.path.exists(Config.DATA_PATH):
        print("Error: Data path not found.")
    else:
        dh = FastDataHandler(Config.DATA_PATH)
        dh.load_data()
        
        fh = FundingHandler(Config.FUNDING_PATH)
        fh.load_rates()
        
        if dh.df_wide is not None:
            
            all_results = {}
            stats_summary = []
            
            for case in TEST_CASES:
                eng = StrategyEngine(
                    dh, fh, 
                    params_name=case["name"],
                    l_freq=case["l_freq"],
                    window_l=case["window_l"],
                    window_y=case["window_y"]
                )
                df_res = eng.run()
                
                if not df_res.empty:
                    capital = Config.MAX_PAIRS * 2
                    metrics = calculate_metrics(df_res, capital)
                    label_str = f"{case['name']}\nRet:{metrics['Total Return']:.0f}% SR:{metrics['Sharpe']:.2f}"
                    df_res['roi'] = (df_res['pnl'] / capital) * 100
                    all_results[label_str] = df_res.set_index('time')['roi']
                    metrics['Name'] = case['name']
                    stats_summary.append(metrics)
                
                gc.collect()

            print("\nGenerating Comparison Chart...")
            plt.figure(figsize=(14, 8))
            
            for label, series in all_results.items():
                plt.plot(series.index, series.values, label=label, linewidth=2)

            try:
                btc = yf.download("BTC-USD", start=Config.START_DATE, end=Config.END_DATE, progress=False)
                if not btc.empty:
                    if isinstance(btc.columns, pd.MultiIndex): close = btc['Close'].iloc[:, 0]
                    else: close = btc['Close']
                    btc_ret = close.pct_change().cumsum() * 100
                    first_res = list(all_results.values())[0]
                    btc_aligned = btc_ret.reindex(first_res.index, method='ffill').fillna(0)
                    if len(btc_aligned) > 0: btc_aligned = btc_aligned - btc_aligned.iloc[0]
                    plt.plot(btc_aligned.index, btc_aligned.values, label="Bitcoin (Benchmark)", 
                             color='black', linestyle='--', alpha=0.6, linewidth=1.5)
            except Exception as e:
                print(f"BTC Download Failed: {e}")

            plt.title('Parameter Sensitivity: Cumulative Return on Capital', fontsize=14, fontweight='bold')
            plt.ylabel('Return (%)', fontsize=12)
            plt.grid(True, alpha=0.5)
            plt.legend(fontsize=10, loc='upper left')
            plt.tight_layout()
            plt.show()
            
            print("\n" + "="*60)
            print(f"{'STRATEGY':<25} | {'RET %':<8} | {'SHARPE':<6} | {'SORTINO':<7} | {'MAX DD %':<8}")
            print("-" * 60)
            for s in stats_summary:
                print(f"{s['Name']:<25} | {s['Total Return']:>7.1f}% | {s['Sharpe']:>6.2f} | {s['Sortino']:>7.2f} | {s['MaxDD']:>7.1f}%")
            print("="*60)